# Wave Equation Factorization via Pseudo-Differential Operators
## From Rigorous Symbolic Validation to Numerical Verification

This notebook demonstrates the factorization of the second-order wave equation into two first-order evolution equations using pseudo-differential operators. 

We approach this in two parts:
1. **Rigorous Symbolic Validation**: We mathematically prove the factorization using asymptotic expansions, highlighting the crucial microlocal corrections required for heterogeneous media.
2. **Numerical Verification**: We numerically solve the 2D wave equation using both the direct second-order formulation and the factored first-order branches, verifying that their sum perfectly reconstructs the original wave field.

---
# Part 1: Rigorous Symbolic Validation

## 1. Rigorous Validation of the Operator Factorization

We claimed that the second-order wave equation can be factored into two first-order equations:
$$
\partial_{tt} u - \Delta u = (\partial_t - P)(\partial_t + P)u = 0
$$
where $P = (-\Delta)^{1/2}$. Mathematically, this requires the pseudo-differential composition to satisfy:
$$
P \circ P = -\Delta
$$

Let's test this identity in two scenarios:
1. **Constant Coefficients (Homogeneous Medium)**: The naive algebraic square root works.
2. **Variable Coefficients (Heterogeneous Medium $c(x)$)**: The naive square root fails, and the `fractional_power` method reveals the hidden microlocal corrections required to save the factorization.

In [ ]:
from psiop import PseudoDifferentialOperator
import sympy as sp
from sympy import symbols, Function, diff, simplify, I, sqrt

x, xi = symbols('x xi', real=True)

### Test A: Constant Coefficients (The Trivial Case)

For a homogeneous medium, $-\Delta$ has the symbol $\xi^2$. 
Its square root is $P = |\xi|$. Let's verify that $P \circ P = \xi^2$.

In [ ]:
# 1. Define the Laplacian symbol (in 1D, -Delta -> xi^2)
Lap_sym = xi**2
Lap_op = PseudoDifferentialOperator(Lap_sym, [x], mode='symbol')

# 2. Compute the fractional power P = (-Delta)^{1/2}
P_sym = Lap_op.fractional_power(alpha=1/2, order=2, method='symbolic')
P_op = PseudoDifferentialOperator(P_sym, [x], mode='symbol')

# 3. Compose P with itself: P∘P
composed_sym = P_op.compose_asymptotic(P_op, order=1, mode='kn')

print("Symbol of P = (-\\Delta)^{1/2}:")
sp.pprint(P_sym)

print("\nSymbol of P∘P:")
sp.pprint(sp.nsimplify(composed_sym))

print("\nDoes P∘P exactly equal \\xi^2 ?", sp.nsimplify(composed_sym - Lap_sym) == 0)

### Test B: Variable Coefficients (The Microlocal Magic)

Now, consider a wave propagating in a heterogeneous medium with spatially varying speed $c(x)$. 
The spatial operator is $L = -c(x)^2 \partial_x^2$, which has the symbol $l(x, \xi) = c(x)^2 \xi^2$.

If we try to factor this naively, we would guess the square root symbol is $p_{\text{naive}} = c(x)\xi$. 
Let's see what happens when we compose this naive symbol with itself using the Kohn-Nirenberg rule:
$$
(p \circ p)(x, \xi) = p^2 - i \partial_\xi p \partial_x p + \dots
$$

Because $c(x)$ and $\xi$ do not commute, the naive factorization breaks!

In [ ]:
# Define spatially varying wave speed c(x)
c = Function('c')(x)

# 1. The true heterogeneous operator L = -c(x)^2 \partial_x^2
L_het_sym = c**2 * xi**2
L_het_op = PseudoDifferentialOperator(L_het_sym, [x], mode='symbol')

# 2. The NAIVE square root guess: p_naive = c(x) * xi
p_naive_sym = c * xi
p_naive_op = PseudoDifferentialOperator(p_naive_sym, [x], mode='symbol')

# 3. Compose the naive symbol with itself
naive_composed = p_naive_op.compose_asymptotic(p_naive_op, order=1, mode='kn')

print("--- NAIVE FACTORIZATION ---")
print("Naive symbol p_naive = c(x) \\xi")
print("Composed p_naive∘p_naive:")
sp.pprint(simplify(naive_composed))

error_term = simplify(naive_composed - L_het_sym)
print(f"\nError (Spurious term generated by non-commutativity):")
sp.pprint(error_term)
print("^ Notice the -i c(x) c'(x) \\xi term! The naive factorization FAILS.")

### The Solution: Asymptotic Fractional Power

To make the factorization $(\partial_t - P)(\partial_t + P) = \partial_t^2 - L$ strictly valid, the symbol of $P$ must include lower-order asymptotic corrections that exactly cancel the spurious error terms generated by the composition.

Let's use our `fractional_power` method to compute the true symbol of $P = L^{1/2}$, and verify that its composition perfectly reconstructs $L$.

In [ ]:
# 1. Compute the TRUE fractional power P = L^{1/2} using asymptotic expansion
P_true_sym = L_het_op.fractional_power(alpha=0.5, order=1, method='symbolic')
P_true_op = PseudoDifferentialOperator(P_true_sym, [x], mode='symbol')

# 2. Compose the TRUE symbol with itself
true_composed = P_true_op.compose_asymptotic(P_true_op, order=1, mode='kn')

print("--- RIGOROUS FACTORIZATION (via fractional_power) ---")
print("True symbol P = L^{1/2} (includes microlocal corrections):")
try:
    sp.pprint(simplify(P_true_sym))
except TypeError:
    sp.pprint(P_true_sym)

print("\n--- Checking the Factorization Error ---")
try:
    diff_expr = simplify(true_composed - L_het_sym)
except TypeError:
    diff_expr = true_composed - L_het_sym

# 💡 CRITICAL STEP: group terms by their ACTUAL power of ξ, not by sympy's
# default term ordering.  `expr.as_ordered_terms()`  sorts by an internal
# monomial heuristic that has nothing to do with powers of ξ, so picking
#  `terms[0]`  can silently drop other terms that sit at the same ξ-order.
#  `.coeff(xi, n)`  is the correct tool: it collects and SUMS every term
# whose ξ-power is exactly n, for each n we care about.
diff_expr_expanded = sp.expand(diff_expr)

print("Coefficients of (P ∘ P - L) by power of ξ:")
coeffs = {}
for n in range(3, -4, -1):          # scan a generous window of powers
    c_n = sp.simplify(diff_expr_expanded.coeff(xi, n))
    if c_n != 0:
        coeffs[n] = c_n
        print(f"\n  O(ξ^{n}):")
        sp.pprint(c_n)

# Sanity check: confirm we've accounted for the entire expression.
# (If this isn't 0, there's a term outside the scanned power range.)
reconstructed = sum(coeffs.get(n, 0) * xi**n for n in coeffs)
residual = sp.simplify(diff_expr_expanded - reconstructed)
if residual != 0:
    print("\n⚠️  Unaccounted residual outside scanned ξ-powers:")
    sp.pprint(residual)

coeff2 = coeffs.get(2, 0)
coeff1 = coeffs.get(1, 0)
leading_error = coeffs.get(0, 0)   # the FULL O(ξ⁰) term, correctly summed

print(f"\nAre the O(ξ²) and O(ξ¹) terms exactly zero?  "
      f"{coeff2 == 0 and coeff1 == 0}")

print(f"\n✅ Full leading error term, O(ξ⁰):")
sp.pprint(leading_error)

print("\n💡 Mathematical Insight:")
if coeff2 == 0 and coeff1 == 0:
    print("The composition proves that the O(ξ²) and O(ξ¹) terms are EXACTLY ZERO!")
    print("The leading error term above is O(ξ⁰), which is the expected asymptotic")
    print("truncation error for an expansion of order=1. The factorization is")
    print("mathematically exact up to the specified asymptotic order!")
else:
    print("Unexpected: a higher-order (ξ² or ξ¹) term did not cancel - check the")
    print("fractional_power / compose_asymptotic implementation for order=1.")

### Conclusion: The Power of Microlocal Calculus

This notebook demonstrated that factoring the wave equation $\partial_{tt} u = c(x)^2 \partial_{xx} u$ into two first-order equations $\partial_t u = \pm P u$ is not a formal algebraic trick, but a genuine pseudodifferential equivalence - one that holds only once $P$ is corrected by a lower-order microlocal term.

1. **The Naive Approach Fails**: Taking the symbol $p_{\text{naive}} = c(x)\xi$ at face value and composing it with itself produces a spurious $-i\,c(x)c'(x)\,\xi$ term - an $O(\xi^1)$ error arising purely from the non-commutativity of multiplication by $c(x)$ and differentiation in $\xi$.

2. **The Rigorous Approach Succeeds**: The asymptotic Newton-type expansion in `fractional_power` finds the correction $P = c(x)\xi + \frac{i}{2}c'(x)$. Composing this corrected symbol with itself exactly cancels the $O(\xi^1)$ term, making the factorization exact through first order.

3. **The WKB Connection**: What remains is a purely $\xi$-independent residual,
$$
\frac{1}{2}c(x)c''(x) - \frac{1}{4}\big(c'(x)\big)^2,
$$
the expected $O(\xi^0)$ truncation error of an order-1 asymptotic expansion, and precisely the classical WKB remainder. The imaginary correction $\frac{i}{2}c'(x)$ is the symbol-level signature of the transport equation that fixes the geometric-optics amplitude $A(x) \propto c(x)^{-1/2}$.

This shows that `psiop`'s symbolic calculus isn't just confirming a known formula - it mechanically derives, term by term, the exact microlocal corrections that a hand derivation would otherwise require.

---
# Part 2: Numerical Verification in 2D
Now that we have rigorously established the symbolic foundation, we numerically verify that the classical 2D wave equation $\partial_{tt} u = \Delta u$ can be perfectly reconstructed by summing the forward and backward first-order branches in a homogeneous medium.

*Key fix*: We use a smaller time step and a wider, smoother Gaussian wave packet to prevent numerical instability and ensure accurate verification.

## 2. Imports

In [ ]:
from solver import PDESolver, psiOp
from psiop import PseudoDifferentialOperator
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output, HTML, Video
from matplotlib.animation import FuncAnimation

## 3. Physical and Simulation Parameters

We define a periodic domain and a sufficiently fine grid to capture the wave dynamics. The time step is kept small (`Nt=1000`) to maintain stability for the fractional Laplacian evolution.

In [ ]:
# ── Grid and Time Parameters ──
Lx, Ly = 4 * np.pi, 4 * np.pi
Nx, Ny = 256, 256
Lt, Nt = 20.0, 1000  # Smaller time step for stability
n_frames = 100

print(f"Domain: [{-Lx/2:.2f}, {Lx/2:.2f}] x [{-Ly/2:.2f}, {Ly/2:.2f}]")
print(f"Grid: {Nx} x {Ny} points")
print(f"Time: {Nt} steps over {Lt} units (dt = {Lt/Nt:.4f})")

## 4. Grid Setup and SymPy Symbols

We set up the spatial mesh and define the symbolic variables required by the `PseudoDifferentialOperator` framework.

In [ ]:
# ── Spatial Mesh ──
xs_1d = np.linspace(-Lx/2, Lx/2, Nx)
ys_1d = np.linspace(-Ly/2, Ly/2, Ny)
xx, yy = np.meshgrid(xs_1d, ys_1d, indexing='ij')

# ── SymPy Symbols ──
t, x, y = sp.symbols('t x y', real=True)
xi, eta = sp.symbols('xi eta', real=True, positive=True)

psi_func = sp.Function('psi')
psi_field = psi_func(t, x, y)
u_func = sp.Function('u')
u_field = u_func(t, x, y)

## 5. Constructing the Fractional Laplacian

The standard wave equation is $\partial_{tt} u - \Delta u = 0$. In Fourier space, $\Delta \to -(\xi^2 + \eta^2)$. 

We first define the standard Laplacian, and then use the `fractional_power` method to compute the symbol of $(-\Delta)^{1/2}$, which is $\sqrt{\xi^2 + \eta^2}$.

In [ ]:
# ── 1. Define the Laplacian ──
symbol_laplacian = -(xi**2 + eta**2)
Lap = PseudoDifferentialOperator(symbol_laplacian, [x, y], mode='symbol')

print("Principal symbol of Laplacian:")
print("  a(ξ, η) = ", symbol_laplacian)

# ── 2. Compute the fractional power (Square Root) ──
sqrt_lap_symbol = Lap.fractional_power(alpha=0.5, order=1, method='symbolic')

print("\nSymbol of the fractional Laplacian √(Δ):")
print("  a_1/2(ξ, η) = ", sqrt_lap_symbol)

# ── 3. Verify the order of the new operator ──
sqrt_lap_op = PseudoDifferentialOperator(sqrt_lap_symbol, [x, y], mode='symbol')
print("\nAsymptotic order of √(Δ):", sqrt_lap_op.symbol_order())

## 6. The Factored First-Order Equations

Using the computed symbol, we construct the forward-propagating ($+$) and backward-propagating ($-$) wave equations:
$$
\partial_t \psi = \pm i \Psi_{\text{op}}\!\left( \sqrt{\xi^2 + \eta^2} \right) \psi
$$

We also define the direct second-order equation for comparison:
$$
\partial_{tt} u = \Psi_{\text{op}}\!\left( -(\xi^2 + \eta^2) \right) u
$$

In [ ]:
# ── Forward and backward branches ──
eq_forward = sp.Eq(sp.diff(psi_field, t), psiOp(sqrt_lap_symbol, psi_field))
eq_backward = sp.Eq(sp.diff(psi_field, t), -1 * psiOp(sqrt_lap_symbol, psi_field))

print("Factored Forward Wave Equation:")
print("  ∂ψ/∂t = psiOp(√(-ξ² - η²), ψ)")
print("\nFactored Backward Wave Equation:")
print("  ∂ψ/∂t = -psiOp(√(-ξ² - η²), ψ)")

# ── Direct second-order equation ──
eq_2nd = sp.Eq(sp.diff(u_field, t, t), psiOp(symbol_laplacian, u_field))

print("\nDirect Second-Order Wave Equation:")
print("  ∂²u/∂t² = psiOp(-(ξ² + η²), u)")

## 7. Initial Conditions: A Localized Wave Packet

We use a wider, smoother Gaussian wave packet to ensure stability and clear visualization. 

**Crucial for Verification**: Since the direct second-order equation starts with zero initial velocity ($\partial_t u(0) = 0$), the energy splits evenly between the forward and backward branches. Therefore, the initial condition for the first-order branches must be exactly half of the second-order initial condition:
$$
\psi_+(0) = \psi_-(0) = \frac{1}{2} u(0)
$$

In [ ]:
def full_ic(xx, yy):
    """Wider, smoother Gaussian wave packet"""
    A0 = 1.0
    W0 = 1.0       # Wider beam waist
    X0 = 0.0       # Centered
    Y0 = 0.0
    KX = 0.0       # Lower carrier frequency
    KY = 0.0
    
    env = np.exp(-((xx - X0)**2 + (yy - Y0)**2) / W0**2)
    phase = KX * xx + KY * yy
    
    return A0 * env * np.exp(1j * phase)

def half_ic(x, y):
    """Half initial condition for the factored branches"""
    return 0.5 * full_ic(x, y)

def zero_vel(x, y):
    """Zero initial velocity for the direct 2nd-order solve"""
    return np.zeros_like(x)

## 8. Solver Setup and Execution

We initialize and run the `PDESolver` for all three formulations: the forward branch, the backward branch, and the direct second-order equation.

In [ ]:
boundary_condition='periodic'
# ── Solve Forward Branch ──
solver_fwd = PDESolver(eq_forward)
solver_fwd.setup(Lx=Lx, Ly=Ly, Nx=Nx, Ny=Ny, Lt=Lt, Nt=Nt,
                 boundary_condition=boundary_condition, initial_condition=half_ic,
                 n_frames=n_frames, plot=False)
frames_fwd = solver_fwd.solve()

# ── Solve Backward Branch ──
solver_bwd = PDESolver(eq_backward)
solver_bwd.setup(Lx=Lx, Ly=Ly, Nx=Nx, Ny=Ny, Lt=Lt, Nt=Nt,
                 boundary_condition=boundary_condition, initial_condition=half_ic,
                 n_frames=n_frames, plot=False)
frames_bwd = solver_bwd.solve()

# ── Solve Direct Second-Order Equation ──
solver_2nd = PDESolver(eq_2nd)
solver_2nd.setup(Lx=Lx, Ly=Ly, Nx=Nx, Ny=Ny, Lt=Lt, Nt=Nt, boundary_condition=boundary_condition, 
                 initial_condition=full_ic, initial_velocity=zero_vel,
                 n_frames=n_frames, plot=False)
frames_2nd = solver_2nd.solve()

print("✅ All solves completed.")

## 9. Verification of the Factorization

We reconstruct the wave field by summing the forward and backward branches:
$$
u_{\text{reconstructed}} = \psi_+ + \psi_-
$$

We then compare this against the direct second-order solution $u_{\text{direct}}$ and compute the maximum absolute error over time.

In [ ]:
# ── Reconstruct and Compare ──
u_reconstructed = np.real(np.asarray(frames_fwd) + np.asarray(frames_bwd))
u_direct = np.real(np.asarray(frames_2nd))

error_per_frame = np.array([
    np.max(np.abs(u_reconstructed[k] - u_direct[k])) for k in range(len(u_direct))
])

# Create a time array corresponding to the saved frames
time_array = np.linspace(0, Lt, len(u_direct))

# ── Plot Error ──
plt.figure(figsize=(8, 5))
plt.plot(time_array, error_per_frame, marker='o', markersize=4, color='blue')
plt.xlabel('Time $t$')
plt.ylabel(r'$\max_{x,y} |(\psi_+ + \psi_-) - u_{\mathrm{direct}}|$')
plt.title('Verification of Wave Equation Factorization')
plt.yscale('log')
plt.grid(True, which="both", ls="--")
plt.tight_layout()
plt.show()

print(f"Final max error at t={Lt:.2f}: {error_per_frame[-1]:.2e}")
print(f"Mean error over time: {np.mean(error_per_frame):.2e}")

## 10. 3D Animation of the Reconstructed Solution

Visualizing the reconstructed wave field $u_{\text{reconstructed}}(x,y,t)$ as a 3D surface plot over time.

In [ ]:
# ── Create animation of the reconstructed solution ──
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

# Pre-calculate time array to match the physical time of each frame
time_array = np.linspace(0, Lt, len(u_reconstructed))

def animate_reconstructed(frame_idx):
    ax.clear()
    surf = ax.plot_surface(xx, yy, u_reconstructed[frame_idx].T, 
                           cmap='viridis', alpha=0.8)
    ax.set_zlim(-1, 1)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_zlabel('u(x,y,t)')
    
    # Reference time instead of frame index
    t_val = time_array[frame_idx]
    ax.set_title(f'Reconstructed Solution - t = {t_val:.2f}')
    
    return [surf]

from matplotlib.animation import FuncAnimation
from IPython.display import HTML

ani_reconstructed = FuncAnimation(fig, animate_reconstructed, 
                                   frames=len(u_reconstructed), 
                                   interval=50, blit=False)
HTML(ani_reconstructed.to_jshtml())

## 11. Side-by-Side Comparison: Reconstructed vs. Direct

To visually confirm the accuracy of the factorization, we plot snapshots of the reconstructed solution, the direct solution, and their difference at selected time intervals.

In [ ]:
# ── Compare reconstructed vs direct solution at selected times ──
n_snapshots = 5
frame_indices = np.linspace(0, len(u_reconstructed)-1, n_snapshots, dtype=int)

fig, axes = plt.subplots(n_snapshots, 3, figsize=(12, 10))

for idx, frame_idx in enumerate(frame_indices):
    t_val = frame_idx * (Lt / n_frames)
    
    # Reconstructed
    im1 = axes[idx, 0].imshow(u_reconstructed[frame_idx].T, 
                               extent=[-Lx/2, Lx/2, -Ly/2, Ly/2],
                               origin='lower', cmap='viridis', 
                               vmin=-1, vmax=1, aspect='equal')
    axes[idx, 0].set_title(f'Reconstructed (t={t_val:.2f})')
    axes[idx, 0].set_ylabel('y')
    plt.colorbar(im1, ax=axes[idx, 0])
    
    # Direct
    im2 = axes[idx, 1].imshow(u_direct[frame_idx].T, 
                               extent=[-Lx/2, Lx/2, -Ly/2, Ly/2],
                               origin='lower', cmap='viridis', 
                               vmin=-1, vmax=1, aspect='equal')
    axes[idx, 1].set_title(f'Direct (t={t_val:.2f})')
    plt.colorbar(im2, ax=axes[idx, 1])
    
    # Difference
    diff = u_reconstructed[frame_idx] - u_direct[frame_idx]
    im3 = axes[idx, 2].imshow(diff.T, 
                               extent=[-Lx/2, Lx/2, -Ly/2, Ly/2],
                               origin='lower', cmap='RdBu', 
                               aspect='equal')
    axes[idx, 2].set_title(f'Difference (t={t_val:.2f})')
    plt.colorbar(im3, ax=axes[idx, 2])
    
    # Calculate and display error norms
    max_error = np.max(np.abs(diff))
    l2_error = np.sqrt(np.mean(diff**2))
    
    # Add text annotation in the top-right corner of the difference plot
    axes[idx, 2].text(0.95, 0.95, 
                      f'max: {max_error:.2e}\nL2: {l2_error:.2e}',
                      transform=axes[idx, 2].transAxes,
                      fontsize=9, verticalalignment='top', 
                      horizontalalignment='right',
                      bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

axes[0, 0].set_ylabel('y')
plt.suptitle('Verification: Reconstructed vs Direct Solution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()